## Deployment

### 1. Persiapan

#### 1.1. Import Library

In [ ]:
import os
import pandas as pd
from config import (
  CLEANED_PROVINCES_CSV,
  CLUSTERED_REGENCIES_CSV,
  SEED_SQL
)

#### 1.2. Persiapan Data

In [ ]:
df_prov = pd.read_csv(CLEANED_PROVINCES_CSV)
df_reg = pd.read_csv(CLUSTERED_REGENCIES_CSV)

### 2. Pembentukan Data Seed SQL Cloudflare D1

#### 2.1. Format Helper SQL Sanitizer

In [ ]:
def sql_val(val):
  if pd.isna(val) or val is None:
    return "NULL"
  if isinstance(val, (int, float)):
    return str(val)
  escaped = str(val).replace("'", "''")
  return f"'{escaped}'"

#### 2.2. Generate Pernyataan SQL Insert

In [ ]:
sql_lines = [
  "-- Cloudflare D1 SQL Seed Generated Automatically by Pipeline\n",
  "-- Insert Provinces"
]

for _, r in df_prov.iterrows():
  p_id = sql_val(r.get('province_id', r.get('no')))
  p_name = sql_val(r['province_name'])
  lat = sql_val(r.get('latitude'))
  lon = sql_val(r.get('longitude'))
  sql_lines.append(
    f"INSERT OR REPLACE INTO provinces (id, name, total_koperasi, koperasi_nib, koperasi_npwp, koperasi_rat, simpanan_pokok, simpanan_wajib, volume_transaksi, nilai_transaksi, latitude, longitude, rasio_nib, rasio_npwp, rasio_rat) VALUES ({p_id}, {p_name}, {r['total_koperasi']}, {r['koperasi_nib']}, {r['koperasi_npwp']}, {r['koperasi_rat']}, {r['simpanan_pokok']}, {r['simpanan_wajib']}, {r['volume_transaksi']}, {r['nilai_transaksi']}, {lat}, {lon}, {r['rasio_nib']}, {r['rasio_npwp']}, {r['rasio_rat']});"
  )

sql_lines.append("\n-- Insert Regencies")
for _, r in df_reg.iterrows():
  r_id = sql_val(f"{r['province_id']}_{r['regency_no']}")
  r_name = sql_val(r['regency_name'])
  lat = sql_val(r.get('latitude'))
  lon = sql_val(r.get('longitude'))
  sql_lines.append(
    f"INSERT OR REPLACE INTO regencies (id, province_id, name, total_koperasi, koperasi_nib, koperasi_npwp, koperasi_rat, simpanan_pokok, simpanan_wajib, volume_transaksi, nilai_transaksi, latitude, longitude, rasio_nib, rasio_npwp, rasio_rat, cluster_label) VALUES ({r_id}, {r['province_id']}, {r_name}, {r['total_koperasi']}, {r['koperasi_nib']}, {r['koperasi_npwp']}, {r['koperasi_rat']}, {r['simpanan_pokok']}, {r['simpanan_wajib']}, {r['volume_transaksi']}, {r['nilai_transaksi']}, {lat}, {lon}, {r['rasio_nib']}, {r['rasio_npwp']}, {r['rasio_rat']}, {r['cluster_label']});"
  )

os.makedirs(os.path.dirname(SEED_SQL), exist_ok=True)
with open(SEED_SQL, 'w', encoding='utf-8') as f:
  f.write("\n".join(sql_lines))

### 3. Verifikasi Output SQL Seed

In [ ]:
file_size_kb = round(os.path.getsize(SEED_SQL) / 1024, 2)
print(f"File SQL Seed berhasil dibuat : {SEED_SQL}")
print(f"Ukuran File                   : {file_size_kb} KB")
print(f"Total Entri Provinsi          : {len(df_prov)}")
print(f"Total Entri Kabupaten/Kota    : {len(df_reg)}")

#### 3.1. Contoh 10 Baris Pertama File SQL

In [ ]:
with open(SEED_SQL, 'r', encoding='utf-8') as f:
  sample_lines = [f.readline().strip() for _ in range(10)]

for line in sample_lines:
  print(line)